# Projeto Retinopatia

Classificação binária de retinopatia diabética em fotos de fundo de olho,
usando transfer learning com EfficientNet.

Este notebook é apenas o **orquestrador da execução no Colab**: toda a lógica
vive nos módulos Python do repositório (`src/retinopatia/`), que são editados
no VS Code e trazidos para cá pelo `git clone`/`git pull` da primeira célula.

**Um experimento por sessão.** O experimento é escolhido na primeira célula
(lista em `src/retinopatia/experimentos.py`). Para rodar experimentos em
paralelo, use uma cópia deste notebook por sessão (Arquivo → Salvar uma cópia
no Drive), mudando só o nome do experimento. Use GPU A100 com RAM alta.

Os experimentos são comparados só na validação; o teste fica fechado até o
modelo final. Depois que as sessões paralelas terminarem, a última célula
consolida o registro de experimentos.

## Preparação do Repositório no Colab

In [ ]:
# O código do projeto vive no repositório do GitHub.
# Esta célula traz (ou atualiza) esse código dentro do Colab.

import os
import sys

URL_REPOSITORIO = "https://github.com/Rcorrea07/ML_Retinopatia.git"
PASTA_REPOSITORIO = "/content/ML_Retinopatia"

# Versão do código a usar. "origin/main" é a mais recente no GitHub.
# Para reproduzir um treino antigo, use a tag ou o commit dele
# (por exemplo "treino-v2"); o commit de cada treino fica no
# configuracao.json da pasta de resultados e no registro de experimentos.
REVISAO_CODIGO = "origin/main"

# Experimento deste treino (lista em src/retinopatia/experimentos.py).
# Escolha antes de rodar; para trocar depois, reinicie o ambiente de
# execução (o pacote já importado guarda a configuração anterior).
EXPERIMENTO = "R01_b0_512"
os.environ["RETINOPATIA_EXPERIMENTO"] = EXPERIMENTO

# "1" = execução de fumaça: poucos pacientes e uma época por fase, para
# conferir em minutos que o experimento roda e salva. Métricas não valem.
os.environ["RETINOPATIA_FUMACA"] = "0"

# "1" só no modelo final escolhido: abre o conjunto de teste
os.environ["RETINOPATIA_AVALIAR_TESTE"] = "0"

# Semente do treino (pesos, embaralhamento e augmentation). A divisão dos
# pacientes não muda (usa sempre a semente 42); repetir um experimento com
# outra semente mede quanto o resultado varia de um treino para outro.
os.environ["RETINOPATIA_SEED_TREINO"] = "42"

if not os.path.exists(PASTA_REPOSITORIO):
    !git clone {URL_REPOSITORIO} {PASTA_REPOSITORIO}

!git -C {PASTA_REPOSITORIO} fetch --tags origin
!git -C {PASTA_REPOSITORIO} checkout --detach {REVISAO_CODIGO}
!git -C {PASTA_REPOSITORIO} log -1 --date=short --format="Código no commit %h (%cd): %s"

# Torna o pacote "retinopatia" importável neste notebook
caminho_src = os.path.join(PASTA_REPOSITORIO, "src")

if caminho_src not in sys.path:
    sys.path.insert(0, caminho_src)

# Recarrega os módulos automaticamente quando esta célula traz código
# novo, evitando ter que reiniciar o ambiente a cada edição no VS Code.
# O Colab usa Python 3.12+, que não tem mais o módulo "imp", mas a
# versão do IPython instalada nele ainda o importa no autoreload.
# Um substituto mínimo resolve; se mesmo assim falhar, o notebook
# segue sem autoreload (aí é preciso reiniciar após cada git pull).
import importlib
import types

try:
    import imp
except ModuleNotFoundError:
    sys.modules["imp"] = types.ModuleType("imp")
    sys.modules["imp"].reload = importlib.reload

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception as erro:
    print("Autoreload indisponível:", erro)

print("Repositório pronto em:", PASTA_REPOSITORIO)

## Configurações iniciais e importações

In [ ]:
from retinopatia import (
    ambiente_colab,
    avaliacao,
    configuracao,
    execucao,
    pre_processamento,
)

# Sementes e precisão mista; vem antes da construção do modelo
execucao.iniciar_execucao()

## Conexão com o Drive

In [ ]:
ambiente_colab.montar_drive()

## Dados do experimento

Imagens do EyePACS (o cache do Drive desta versão e deste tamanho, se existir; senão,
extração e pré-processamento offline, ~50 min), rótulos e divisão por paciente
(treino, validação e teste, estratificada pelo grau). Depois, as imagens que só entram
no treino: as 53.576 do "test" da competição de 2015, com os rótulos publicados
depois pelos organizadores, e os datasets externos que o experimento escolher
(`DATASETS_EXTERNOS`: DDR, APTOS, DeepDRiD, IDRiD; ver `docs/DATASETS.md`). Por fim,
os ajustes do experimento: execução de fumaça (poucos pacientes), outro olho de cada
linha (modelo de pares) e probabilidades do professor no treino (destilação, que usa
a GPU e libera a memória do professor antes do modelo). Validação e teste não mudam.

In [ ]:
conjuntos = execucao.preparar_dados()

## Inspeção visual

Foto original, máscara da retina e imagem processada, numa amostra e nas imagens com
máscara suspeita. Só quando as fotos originais foram extraídas nesta sessão.

In [ ]:
pre_processamento.pre_visualizar_recorte(conjuntos.dados)

In [ ]:
status = conjuntos.status_imagens
suspeitas = status[status["status"] == configuracao.STATUS_SUSPEITO]["image"]

pre_processamento.pre_visualizar_recorte(conjuntos.dados, imagens=suspeitas)

## Treino

Pipelines (com a conferência da escala e a velocidade do pipeline), pesos das classes,
modelo, configuração gravada no Drive, Fase 1 (só as cabeças, com a avaliação dela na
validação), Fase 2 (fine-tuning) e o modelo completo salvo e conferido.

In [ ]:
treino = execucao.treinar(conjuntos, avaliar_fase1=True, medir_velocidade=True)

## Avaliação, teste e diagnóstico

Fase 2 na validação (limiares por olho e por paciente, modos de TTA); o teste só com
`RETINOPATIA_AVALIAR_TESTE=1` (modelo final); o resumo da execução (`resumo.csv`); e o
diagnóstico: erros por grau, retinopatia referenciável, cabeça de grau, agregação dos
olhos e intervalos de confiança. Nos experimentos, tudo é medido na validação
(otimista, mas comparável entre experimentos); no modelo final, as partes de erro e
intervalo usam o teste. As tabelas são gravadas na pasta do experimento.

In [ ]:
resultado = execucao.avaliar_e_registrar(treino, conjuntos)

## Registro de todos os experimentos

Rode em **uma** sessão só, depois que as sessões paralelas terminarem: junta os
`resumo.csv` de cada execução em `resultados/registro_experimentos.csv`.

In [ ]:
registro = avaliacao.consolidar_experimentos()